# The Visual Storyteller — Inference & Analysis

**Notebook 2 of 2** (Person 2 lead). Demonstrates the trained captioning models on **unseen test images** and analyses where they succeed and fail.

Contents:
1. Setup & load artifacts (vocab + trained models)
2. The graded entry point: `generate_caption(image_path, model) -> str`
3. Demonstration on held-out test images
4. Success cases
5. Failure cases (diagnosed)
6. Baseline vs Transformer comparison table

> All logic lives in `src/`; this notebook stays thin. Run **Restart & Run All** on a clean runtime before submitting.

## 1. Setup

On Colab, mount Drive and `cd` into the repo first. Locally, just run from the repo root.

In [ ]:
import os, sys

# --- Colab setup (uncomment on Colab) ---
# from google.colab import drive; drive.mount('/content/drive')
# %cd /content/visual_storyteller

# Make `src` importable when running from notebooks/
REPO = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import torch
import matplotlib.pyplot as plt
from PIL import Image

from src.config import default_config
from src.data import Vocabulary, load_captions, references_for, make_splits, load_official_splits
from src.decode import generate_caption, greedy_decode, beam_search
from src import evaluate as E

cfg = default_config()  # use cfg.use_colab_paths(...) on Colab
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print('checkpoint dir:', cfg.checkpoint_dir)

## 2. Load artifacts

Loads the vocabulary and the two trained models. The model class comes from Person 1's `src/model.py` (`EncoderDecoder`). We attach the few attributes `generate_caption` needs (`vocab`, `device`, `encoder`, `image_transform`, `beam_size`) so its signature stays exactly `(image_path, model) -> str`.

> If `src/model.py` / `src/encoder.py` aren't merged yet, this cell prints a clear notice and the demo cells below are skipped — the rest of the notebook (vocab, references, table formatting) still runs.

In [ ]:
vocab = Vocabulary.load(cfg.vocab_file)
print('vocab size:', len(vocab))

MODELS = {}
MODELS_READY = False
try:
    from src.model import EncoderDecoder          # Person 1's wrapper
    from src.encoder import build_image_transform  # Person 1's eval transform

    def load_model(ckpt_name, variant):
        model = EncoderDecoder(vocab_size=len(vocab), cfg=cfg, variant=variant).to(device)
        ckpt = torch.load(os.path.join(cfg.checkpoint_dir, ckpt_name), map_location=device)
        model.load_state_dict(ckpt['model'])
        model.eval()
        # attributes generate_caption() reads off the model object
        model.vocab = vocab
        model.device = device
        model.image_transform = build_image_transform()
        model.beam_size = cfg.beam_size
        model.max_len = cfg.max_caption_len
        return model

    MODELS['baseline'] = load_model('baseline_best.pt', 'lstm')
    MODELS['transformer'] = load_model('transformer_best.pt', 'transformer')
    MODELS_READY = True
    print('loaded:', list(MODELS))
except (ImportError, FileNotFoundError) as e:
    print('[notice] models not available yet — demo cells will be skipped.')
    print('        reason:', type(e).__name__, e)

## 3. The graded entry point

`generate_caption(image_path: str, model: any) -> str` — the exact signature the brief asks for. It loads the image, runs the frozen encoder, beam-searches, and detokenises to a sentence. Defined in `src/decode.py`.

In [ ]:
# Held-out test images — untouched during development.
captions = load_captions(cfg.captions_file)
splits = load_official_splits(cfg.data_dir) or make_splits(list(captions), seed=cfg.seed)
test_ids = splits['test']
test_refs = references_for(test_ids, captions)
print(f'{len(test_ids)} held-out test images')

def show(image_id, caps):
    """Display a test image with model captions + a reference."""
    path = os.path.join(cfg.images_dir, image_id)
    plt.figure(figsize=(5, 5)); plt.imshow(Image.open(path)); plt.axis('off')
    title = '\n'.join(f'{k}: {v}' for k, v in caps.items())
    ref = ' '.join(test_refs[image_id][0]) if test_refs.get(image_id) else ''
    plt.title(title + (f'\nref: {ref}' if ref else ''), fontsize=9, loc='left')
    plt.show()

def caption_all(image_id):
    path = os.path.join(cfg.images_dir, image_id)
    return {name: generate_caption(path, m) for name, m in MODELS.items()}

In [ ]:
if MODELS_READY:
    demo_id = test_ids[0]
    print('image:', demo_id)
    print(caption_all(demo_id))
else:
    print('skipped — models not loaded')

## 4. Success cases

~8 strong examples where the models produce accurate, fluent captions. Pick a spread of scenes (people, animals, action, outdoor/indoor).

In [ ]:
# Curate after a first pass over the test set; placeholder picks the first few.
SUCCESS_IDS = test_ids[:8]

if MODELS_READY:
    for img_id in SUCCESS_IDS:
        show(img_id, caption_all(img_id))
else:
    print('skipped — models not loaded')

## 5. Failure cases (diagnosed)

~5 cases where captions go wrong. For each, note the **failure type**: object hallucination, miscounting, colour/scene error, repetition, or over-generic caption. These diagnoses are high-value for the grade.

In [ ]:
# (image_id, diagnosis) — fill in after inspecting outputs.
FAILURE_CASES = [
    # ('1234567890_abc.jpg', 'object hallucination: invents a dog not in the image'),
    # ('...jpg', 'miscount: says two when there are three'),
]

if MODELS_READY:
    for img_id, diagnosis in FAILURE_CASES:
        caps = caption_all(img_id); caps['DIAGNOSIS'] = diagnosis
        show(img_id, caps)
else:
    print('skipped — models not loaded')

## 6. Baseline vs Transformer — quantitative comparison

Corpus BLEU-1..4 over the full held-out test set against all 5 references, for each model and decoding strategy. Goes into `reports/results.md` and the README.

In [ ]:
def evaluate_model(model, image_ids, use_beam=True):
    """Caption every test image, return {image_id: hyp_tokens}."""
    hyps = {}
    for img_id in image_ids:
        path = os.path.join(cfg.images_dir, img_id)
        feats = model.encoder(model.image_transform(
            Image.open(path).convert('RGB')).unsqueeze(0).to(device)).squeeze(0)
        if use_beam:
            ids = beam_search(model, feats, vocab, beam_size=cfg.beam_size,
                              max_len=cfg.max_caption_len, device=device)
        else:
            ids = greedy_decode(model, feats.unsqueeze(0), vocab,
                                max_len=cfg.max_caption_len, device=device)[0]
        hyps[img_id] = vocab.decode(ids)
    return hyps

if MODELS_READY:
    rows = {}
    for name, model in MODELS.items():
        for strat, beam in [('greedy', False), ('beam', True)]:
            hyps = evaluate_model(model, test_ids, use_beam=beam)
            rows[f'{name}-{strat}'] = E.score_bleu(hyps, test_refs)
    table = E.format_score_table(rows)
    print(table)
    os.makedirs(os.path.join(REPO, 'reports'), exist_ok=True)
    with open(os.path.join(REPO, 'reports', 'results.md'), 'w', encoding='utf-8') as f:
        f.write('# Results — BLEU on held-out test set\n\n' + table + '\n')
    print('\nwrote reports/results.md')
else:
    print('skipped — models not loaded')